[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/08_Kalman_Filter.ipynb)

# DiveLab

## Notebook 08 — Kalman Filter: Estimating Depth and Vertical Velocity

**Guiding question:** How can we combine an imperfect physical model with a noisy depth sensor to estimate the diver's state?

> A Kalman filter is a systematic way to combine **what the model predicts** with **what the sensor measures**.

## Learning objectives

By the end of this lab, you will be able to:

- explain a Kalman filter in simple words;
- distinguish prediction, measurement and estimate;
- distinguish process noise from measurement noise;
- implement the predict–correct cycle;
- understand covariance as uncertainty;
- interpret the Kalman gain;
- estimate vertical velocity from noisy depth measurements;
- study what happens when we trust the sensor or the model too much.

## From Notebook 07 to Notebook 08

Notebook 07 introduced an observer:

$$
\dot{\hat x}=A\hat x+L(y-C\hat x)
$$

The observer combines:

- a model prediction;
- a measurement correction.

But we chose the observer gain $L$ ourselves.

Now we ask:

> Can the estimator decide automatically how strongly to trust the model and how strongly to trust the sensor?

That is the central idea of the **Kalman filter**.

# What is a Kalman filter?

Imagine that you want to know the diver's depth and vertical velocity.

You have two imperfect sources of information.

### 1. The physical model

The model predicts how the diver should move.

But the model is never perfect.

It may ignore:

- breathing;
- fin movement;
- currents;
- imperfect drag parameters;
- small buoyancy changes.

### 2. The depth sensor

The sensor measures depth directly.

But its measurement is noisy.

The Kalman filter combines these two sources.

In simple words:

> **Predict where the diver should be, look at what the sensor says, then correct the prediction by an amount that depends on how much you trust each source.**

It repeats this process at every timestep.

## A simple analogy

Suppose the model predicts:

$$
20.2\ \mathrm{m}
$$

but the sensor reports:

$$
19.8\ \mathrm{m}
$$

Should we believe the model or the sensor?

The answer depends on uncertainty.

If the sensor is very accurate, the estimate should move strongly toward 19.8 m.

If the sensor is very noisy but the model is reliable, the estimate should stay closer to 20.2 m.

The Kalman filter computes this compromise mathematically.

# The three quantities

As in Notebook 07:

$$
x_k
$$

is the true state,

$$
y_k
$$

is the measurement,

and:

$$
\hat x_k
$$

is our estimate.

For this notebook:

$$
x_k=
\begin{bmatrix}
\delta z_k\\
\delta v_k
\end{bmatrix}
$$

but the sensor measures only depth:

$$
y_k=\delta z_k+\text{noise}.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Part 1 — A discrete state-space model

A Kalman filter is naturally written in discrete time:

$$
x_{k+1}=F x_k+w_k
$$

$$
y_k=H x_k+v_k
$$

where:

- $F$ is the state-transition matrix;
- $H$ maps the state to the measured output;
- $w_k$ is process noise;
- $v_k$ is measurement noise.

## Start from the DiveLab linearized model

Near neutral buoyancy:

$$
\delta\dot z=-\delta v
$$

$$
\delta\dot v=a_z\delta z
$$

or:

$$
\dot x=Ax.
$$

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
gas_surface_volume_e = 0.005

def pressure_at_depth(z):
    return P0 + rho * g * z

dFb_dz = (
    -rho * g
    * gas_surface_volume_e
    * P0
    * rho * g
    / pressure_at_depth(z_e)**2
)

a_z = dFb_dz / mass

A = np.array([
    [0.0, -1.0],
    [a_z, 0.0]
])

print("Continuous-time A:")
print(A)
print("Eigenvalues:", np.linalg.eigvals(A))

## Discretization

For a small timestep $\Delta t$, a simple approximation is:

$$
F\approx I+A\Delta t.
$$

This is the Euler discretization.

For teaching purposes it lets us see clearly how the continuous model becomes a discrete estimator.

In [ ]:
dt = 0.05

F = np.eye(2) + A * dt

H = np.array([
    [1.0, 0.0]
])

print("F =")
print(F)
print()
print("H =")
print(H)

Because:

$$
H=
\begin{bmatrix}
1&0
\end{bmatrix},
$$

the sensor measures depth deviation but does not measure vertical velocity directly.

The Kalman filter will estimate both.

# Part 2 — Two kinds of uncertainty

This distinction is fundamental.

## Measurement noise

Measurement noise describes uncertainty in the sensor:

$$
v_k.
$$

Its covariance is:

$$
R.
$$

Large $R$ means:

> "I do not trust the sensor very much."

## Process noise

Process noise describes uncertainty in the model:

$$
w_k.
$$

Its covariance is:

$$
Q.
$$

Large $Q$ means:

> "I do not trust the model very much."

## Why process noise matters

Even if our equations were mathematically exact, the simplified model does not include everything that affects a real diver.

For example:

> breathing → buoyancy variation → acceleration disturbance

The Kalman filter represents such unmodeled effects statistically through $Q$.

In [ ]:
# Process-noise covariance
Q = np.array([
    [1e-5, 0.0],
    [0.0, 5e-5]
])

# Depth sensor standard deviation
sigma_sensor = 0.08

# Measurement-noise covariance
R = np.array([
    [sigma_sensor**2]
])

print("Q =")
print(Q)
print()
print("R =")
print(R)

# Part 3 — Simulate a true trajectory

We create a hidden true state and add random disturbances.

The estimator will not be allowed to see this true state.

It sees only the noisy depth measurement.

In [ ]:
duration = 25.0
n = int(duration / dt) + 1
t = np.linspace(0, duration, n)

x_true = np.zeros((n, 2))
y_meas = np.zeros(n)

x_true[0] = np.array([0.0, 0.05])

# Slightly stronger disturbances than the estimator assumes,
# to make the estimation problem visible.
Q_true = np.array([
    [2e-5, 0.0],
    [0.0, 8e-5]
])

for k in range(n - 1):
    w = rng.multivariate_normal(np.zeros(2), Q_true)
    x_true[k + 1] = F @ x_true[k] + w

for k in range(n):
    measurement_noise = rng.normal(0.0, sigma_sensor)
    y_meas[k] = (H @ x_true[k])[0] + measurement_noise

In [ ]:
plt.plot(t, x_true[:, 0], label="True depth deviation")
plt.plot(t, y_meas, alpha=0.45, label="Measured depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Hidden true depth and noisy measurement")
plt.grid(True)
plt.legend()
plt.show()

The sensor data are noisy, but the true state evolves according to a dynamical model.

The Kalman filter will exploit both facts.

# Part 4 — The Kalman filter cycle

Every timestep has two phases:

## Phase A — Predict

Predict the next state:

$$
\hat x_k^- = F\hat x_{k-1}
$$

Predict the uncertainty:

$$
P_k^- = FP_{k-1}F^T+Q.
$$

The superscript $-$ means:

> estimate before seeing the new measurement.

## What is $P$?

$P$ is the **state-estimation error covariance matrix**.

In simple words:

> $P$ represents how uncertain the filter believes its state estimate is.

For our two-state system:

$$
P=
\begin{bmatrix}
P_{zz} & P_{zv}\\
P_{vz} & P_{vv}
\end{bmatrix}.
$$

The diagonal terms describe uncertainty in depth and velocity.

The off-diagonal terms describe how the two estimation errors are related.

## Phase B — Correct

First compute the innovation:

$$
r_k=y_k-H\hat x_k^-.
$$

This is:

> sensor measurement − predicted measurement.

Then compute the innovation covariance:

$$
S_k=HP_k^-H^T+R.
$$

Then the Kalman gain:

$$
K_k=P_k^-H^TS_k^{-1}.
$$

Finally correct the state:

$$
\hat x_k=\hat x_k^-+K_kr_k
$$

and update its uncertainty:

$$
P_k=(I-K_kH)P_k^-.
$$

# What is the Kalman gain?

The Kalman gain $K_k$ decides how strongly the new measurement should correct the prediction.

Conceptually:

### Large Kalman gain

The filter says:

> "The sensor is useful relative to my current prediction uncertainty. Correct the estimate strongly."

### Small Kalman gain

The filter says:

> "The prediction is relatively trustworthy, or the sensor is noisy. Correct only a little."

The gain is recomputed as uncertainty evolves.

## An important clarification

The Kalman gain is not simply:

> sensor good → large gain.

It depends on the **relative uncertainty** of prediction and measurement.

The filter asks:

> How uncertain am I about the predicted state compared with how uncertain the sensor is?

# Part 5 — Implement the Kalman filter

We will write the algorithm explicitly rather than use a library.

This lets us see every equation.

In [ ]:
def kalman_filter(y, F, H, Q, R, xhat0, P0):
    n = len(y)
    nx = len(xhat0)

    xhat = np.zeros((n, nx))
    xpred = np.zeros((n, nx))

    P_hist = np.zeros((n, nx, nx))
    K_hist = np.zeros((n, nx))
    innovation = np.zeros(n)

    xhat[0] = xhat0
    P = P0.copy()
    P_hist[0] = P

    I = np.eye(nx)

    for k in range(1, n):

        # -------------------
        # 1. PREDICT
        # -------------------
        x_minus = F @ xhat[k - 1]
        P_minus = F @ P @ F.T + Q

        # Predicted measurement
        y_minus = (H @ x_minus)[0]

        # -------------------
        # 2. INNOVATION
        # -------------------
        r = y[k] - y_minus
        S = (H @ P_minus @ H.T + R)[0, 0]

        # -------------------
        # 3. KALMAN GAIN
        # -------------------
        K = (P_minus @ H.T)[:, 0] / S

        # -------------------
        # 4. CORRECT
        # -------------------
        x_plus = x_minus + K * r

        # Joseph-form covariance update:
        # numerically more robust than P = (I-KH)P_minus
        KH = np.outer(K, H[0])
        P = (
            (I - KH) @ P_minus @ (I - KH).T
            + np.outer(K, K) * R[0, 0]
        )

        xpred[k] = x_minus
        xhat[k] = x_plus
        P_hist[k] = P
        K_hist[k] = K
        innovation[k] = r

    return xhat, xpred, P_hist, K_hist, innovation

## Initial estimate

Suppose the filter starts with the wrong idea about the state:

$$
\hat x_0=
\begin{bmatrix}
0.4\\
-0.10
\end{bmatrix}.
$$

We also tell the filter that this initial estimate is uncertain through $P_0$.

In [ ]:
xhat0 = np.array([0.4, -0.10])

P0 = np.array([
    [0.25, 0.0],
    [0.0, 0.10]
])

x_hat, x_pred, P_hist, K_hist, innovation = kalman_filter(
    y=y_meas,
    F=F,
    H=H,
    Q=Q,
    R=R,
    xhat0=xhat0,
    P0=P0
)

# Part 6 — Estimated depth

In [ ]:
plt.plot(t, x_true[:, 0], label="True depth")
plt.plot(t, y_meas, alpha=0.25, label="Sensor")
plt.plot(t, x_hat[:, 0], label="Kalman estimate")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Kalman-filtered depth estimate")
plt.grid(True)
plt.legend()
plt.show()

Notice the three levels:

- the **true depth** is hidden;
- the **measurement** is noisy;
- the **estimate** is smoother but still follows the real motion.

The filter does not merely smooth the sensor.

It uses the system dynamics to decide how the state should evolve.

# Part 7 — Estimate an unmeasured velocity

The depth sensor never measures velocity.

Yet the Kalman filter can estimate:

$$
\delta v.
$$

This is possible because the system is observable, as we showed in Notebook 07.

In [ ]:
plt.plot(t, x_true[:, 1], label="True vertical velocity")
plt.plot(t, x_hat[:, 1], label="Kalman velocity estimate")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity deviation [m/s]")
plt.title("Estimating an unmeasured state")
plt.grid(True)
plt.legend()
plt.show()

This is a key idea:

> **The Kalman filter is not only a noise filter. It is a state estimator.**

It can reconstruct hidden states by combining measurements with a model.

# Part 8 — Prediction and correction

Let's zoom in on what the filter does.

Before receiving the measurement it has:

$$
\hat x_k^-.
$$

After processing the measurement it has:

$$
\hat x_k.
$$

The difference is the correction.

In [ ]:
plt.plot(t, x_true[:, 0], label="True depth")
plt.plot(t, x_pred[:, 0], alpha=0.8, label="Prediction")
plt.plot(t, x_hat[:, 0], label="Corrected estimate")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Kalman cycle: predict, then correct")
plt.grid(True)
plt.legend()
plt.show()

At every sample:

> model predicts → sensor arrives → discrepancy is measured → prediction is corrected.

# Part 9 — Innovation

The innovation is:

$$
r_k=y_k-H\hat x_k^-.
$$

It is the difference between:

- what the sensor says;
- what the filter expected the sensor to say.

In [ ]:
plt.plot(t, innovation)

plt.xlabel("Time [s]")
plt.ylabel("Innovation [m]")
plt.title("Kalman innovation")
plt.grid(True)
plt.show()

A well-behaved innovation should fluctuate around zero when:

- the model is reasonable;
- the sensor model is reasonable;
- there is no persistent bias or fault.

A sustained non-zero innovation can be a clue that something has changed.

This links Kalman filtering back to the sensor-failure discussion in Notebook 06.

# Part 10 — Watch the Kalman gain evolve

In [ ]:
plt.plot(t, K_hist[:, 0], label="Depth gain")
plt.plot(t, K_hist[:, 1], label="Velocity gain")

plt.xlabel("Time [s]")
plt.ylabel("Kalman gain")
plt.title("Evolution of the Kalman gain")
plt.grid(True)
plt.legend()
plt.show()

Initially the filter may be uncertain because $P_0$ is large.

As measurements accumulate, its uncertainty changes and the gain evolves.

For a time-invariant observable system with constant $Q$ and $R$, the gain often approaches a steady value.

# Part 11 — Watch uncertainty evolve

The standard deviation associated with each estimated state is approximately:

$$
\sigma_z=\sqrt{P_{zz}}
$$

$$
\sigma_v=\sqrt{P_{vv}}.
$$

In [ ]:
sigma_z_est = np.sqrt(P_hist[:, 0, 0])
sigma_v_est = np.sqrt(P_hist[:, 1, 1])

plt.plot(t, sigma_z_est, label="Estimated depth uncertainty")
plt.plot(t, sigma_v_est, label="Estimated velocity uncertainty")

plt.xlabel("Time [s]")
plt.ylabel("Estimated standard deviation")
plt.title("Kalman estimate uncertainty")
plt.grid(True)
plt.legend()
plt.show()

This is another major advantage of the Kalman filter:

> it estimates not only the state, but also how uncertain that estimate is.

# Part 12 — What if the sensor becomes noisier?

Increase $R$.

That tells the filter:

> trust the depth sensor less.

In [ ]:
R_noisy = np.array([
    [(0.30)**2]
])

x_hat_noisyR, _, _, K_noisyR, _ = kalman_filter(
    y=y_meas,
    F=F,
    H=H,
    Q=Q,
    R=R_noisy,
    xhat0=xhat0,
    P0=P0
)

plt.plot(t, x_true[:, 0], label="True depth")
plt.plot(t, x_hat[:, 0], label="Baseline R")
plt.plot(t, x_hat_noisyR[:, 0], label="Larger R")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Increasing R: trust the sensor less")
plt.grid(True)
plt.legend()
plt.show()

With larger $R$, the estimate tends to rely more heavily on the model.

The corresponding measurement correction becomes weaker.

# Part 13 — What if the model is less trustworthy?

Increase $Q$.

That tells the filter:

> unexpected state changes are more plausible, so trust the model prediction less.

In [ ]:
Q_large = np.array([
    [5e-4, 0.0],
    [0.0, 2e-3]
])

x_hat_largeQ, _, _, K_largeQ, _ = kalman_filter(
    y=y_meas,
    F=F,
    H=H,
    Q=Q_large,
    R=R,
    xhat0=xhat0,
    P0=P0
)

plt.plot(t, x_true[:, 0], label="True depth")
plt.plot(t, x_hat[:, 0], label="Baseline Q")
plt.plot(t, x_hat_largeQ[:, 0], label="Larger Q")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Increasing Q: trust the model less")
plt.grid(True)
plt.legend()
plt.show()

## The central tuning idea

Very roughly:

### Increase $R$

> "The sensor is noisy."

The filter trusts measurements less.

### Increase $Q$

> "The model is uncertain."

The filter becomes more willing to follow measurements and unexpected motion.

So $Q$ and $R$ encode our assumptions about uncertainty.

# Part 14 — Quantify estimation performance

Let's compute root mean squared error:

$$
RMSE=
\sqrt{
\frac{1}{N}
\sum_{k=1}^{N}
(x_k-\hat x_k)^2
}.
$$

In [ ]:
depth_sensor_rmse = np.sqrt(
    np.mean((y_meas - x_true[:, 0])**2)
)

depth_kalman_rmse = np.sqrt(
    np.mean((x_hat[:, 0] - x_true[:, 0])**2)
)

velocity_kalman_rmse = np.sqrt(
    np.mean((x_hat[:, 1] - x_true[:, 1])**2)
)

print(f"Raw depth sensor RMSE:      {depth_sensor_rmse:.4f} m")
print(f"Kalman depth RMSE:          {depth_kalman_rmse:.4f} m")
print(f"Kalman velocity RMSE:       {velocity_kalman_rmse:.4f} m/s")

RMSE gives us an objective way to compare estimator configurations.

But remember:

> the Kalman filter is optimal only relative to its model and statistical assumptions.

# Part 15 — What does "optimal" mean?

Under the standard linear-Gaussian assumptions, the Kalman filter minimizes the mean-square estimation error.

That does **not** mean:

- the model is always correct;
- the estimate is always correct;
- every real sensor behaves Gaussian;
- every nonlinear system can be handled perfectly by this filter.

It means that, for the assumed linear model and noise statistics, the filter gives a mathematically optimal linear estimate in the mean-square sense.

# Part 16 — Kalman filter vs low-pass filter

A low-pass filter mainly asks:

> How can I smooth this signal?

A Kalman filter asks:

> Given a dynamical model, a measurement, and uncertainty in both, what is my best estimate of the state?

That difference is important.

A Kalman filter can estimate quantities that are not directly measured, such as vertical velocity.

# Part 17 — Kalman filter vs Luenberger observer

The structures are closely related.

### Luenberger observer

$$
\dot{\hat x}
=
A\hat x
+
L(y-C\hat x)
$$

We choose $L$ to obtain desired observer dynamics.

### Kalman filter

The correction also depends on:

$$
y-H\hat x^-,
$$

but the gain is computed from uncertainty:

$$
K=P^-H^T(HP^-H^T+R)^{-1}.
$$

So the Kalman filter can be viewed as a model-based observer whose gain is derived from stochastic uncertainty.

# Part 18 — The complete information loop

We can now extend our DiveLab architecture:

> physical diver → depth sensor → Kalman filter → state estimate → controller → BCD → physical diver

The controller can use:

$$
\hat x=
\begin{bmatrix}
\hat z\\
\hat v
\end{bmatrix}
$$

instead of requiring perfect access to:

$$
x.
$$

## Sensor failure revisited

A Kalman filter does not magically solve sensor failure.

If the sensor develops:

- bias;
- drift;
- frozen readings;
- large outliers;

the assumptions behind the filter may become invalid.

However, the innovation provides useful diagnostic information.

For example, a persistent innovation may indicate that:

- the model is wrong;
- the sensor is biased;
- an external disturbance is present.

# Part 19 — A depth-sensor bias experiment

Let's add a constant bias after 12 seconds.

In [ ]:
y_bias = y_meas.copy()
y_bias[t >= 12.0] += 0.4

x_hat_bias, _, _, _, innovation_bias = kalman_filter(
    y=y_bias,
    F=F,
    H=H,
    Q=Q,
    R=R,
    xhat0=xhat0,
    P0=P0
)

plt.plot(t, x_true[:, 0], label="True depth")
plt.plot(t, y_bias, alpha=0.25, label="Biased sensor")
plt.plot(t, x_hat_bias[:, 0], label="Kalman estimate")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Kalman filter with sensor bias")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t, innovation_bias)
plt.axvline(12.0, linestyle="--", label="Bias introduced")

plt.xlabel("Time [s]")
plt.ylabel("Innovation [m]")
plt.title("Innovation after sensor bias")
plt.grid(True)
plt.legend()
plt.show()

The filter assumes zero-mean measurement noise.

A persistent sensor bias violates that assumption.

This is why estimation and fault detection belong together.

# Part 20 — Systems-theory view

The Kalman filter brings together several ideas from previous notebooks:

### Dynamics

$$
x_{k+1}=Fx_k+w_k
$$

### Measurement

$$
y_k=Hx_k+v_k
$$

### Observability

The measured output must contain enough information to reconstruct the hidden state.

### Prediction

Use the plant model.

### Innovation

Compare prediction with measurement.

### Correction

Use the innovation to update the estimate.

### Uncertainty

Represent uncertainty through covariance matrices.

### Feedback

The estimated state can be fed to the controller.

## The predict–correct loop in one picture

Conceptually:

```text
             MODEL
               |
               v
previous --> PREDICT --> predicted state
 estimate                  |
                           v
sensor -------> COMPARE --> innovation
                           |
                           v
                         CORRECT
                           |
                           v
                     new state estimate
```

Then the cycle repeats.

# Exercises

### 1. Change sensor noise

Try:

```python
sigma_sensor = 0.02
sigma_sensor = 0.15
sigma_sensor = 0.50
```

Update $R$ consistently.

How do the estimate and Kalman gain change?

### 2. Change process noise

Multiply $Q$ by:

```python
0.1
1
10
100
```

When does the filter begin to follow the sensor very closely?

### 3. Wrong initial state

Try:

```python
xhat0 = np.array([1.0, -0.5])
```

How quickly does the filter recover?

### 4. Wrong initial uncertainty

Compare:

```python
P0 = 0.001 * np.eye(2)
```

with:

```python
P0 = 10 * np.eye(2)
```

What does each choice tell the filter about its initial confidence?

### 5. Velocity estimation

Compare the Kalman velocity estimate with a raw numerical derivative of the noisy depth measurement.

Which one is more useful?

## Challenge — sensor fault detection

Use the innovation:

$$
r_k=y_k-H\hat x_k^-
$$

and its expected uncertainty:

$$
S_k=HP_k^-H^T+R
$$

to construct a normalized residual.

Can you detect the sensor bias introduced at 12 seconds?

This leads naturally toward **innovation-based fault detection**.

In [ ]:
# Your code here

# Summary

In this lab we learned that a Kalman filter:

- predicts the state using a model;
- predicts how uncertain that state estimate is;
- compares the prediction with the sensor;
- computes an innovation;
- computes a Kalman gain;
- corrects the state estimate;
- updates its uncertainty;
- can estimate states that are not measured directly.

### In simple words

> **The Kalman filter continuously negotiates between model and sensor.**

If the model is uncertain, it listens more to the measurement.

If the sensor is noisy, it relies more on the model.

And it continually updates how confident it is.

### Core equations

Predict:

$$
\hat x_k^- = F\hat x_{k-1}
$$

$$
P_k^- = FP_{k-1}F^T+Q
$$

Correct:

$$
r_k=y_k-H\hat x_k^-
$$

$$
K_k=P_k^-H^T(HP_k^-H^T+R)^{-1}
$$

$$
\hat x_k=\hat x_k^-+K_kr_k
$$

$$
P_k=(I-K_kH)P_k^-
$$

### Next

Notebook 09 can close the loop:

> **Kalman Filter + Feedback Controller**

We can compare perfect-state feedback with realistic estimated-state feedback and introduce the **LQG architecture**.